<a href="https://colab.research.google.com/github/Sarang68/confluence-to-vectordb/blob/main/Confluence_to_VectorDB_Three_Approaches.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Confluence → Vector Database
## Three Production Approaches — Colab Ready

| Approach | Tool | Best for |
|---|---|---|
| **A** | LangChain `ConfluenceLoader` | Prototypes, <500 pages, 20 lines |
| **B** | Confluence REST API direct | Production, incremental sync, full metadata control |
| **C** | Vertex AI RAG Engine (managed) | GCP-native, no infra, built-in grounding |

**Running example:** All three approaches work on the same mock Confluence pages so you can compare output side-by-side. Swap in real credentials to point at your actual Confluence instance.

---
**Key concepts this lab covers:**
- CQL (Confluence Query Language) for filtered + incremental fetches  
- HTML → clean text conversion preserving table structure  
- Breadcrumb context injection into chunks before embedding  
- `content_hash`-based incremental sync (only re-index changed pages)  
- Metadata pre-filtering at retrieval time (space_key, labels)

## ⚙️ Setup

In [ ]:
!pip install -q google-cloud-aiplatform>=1.60.0
!pip install -q langchain-google-vertexai langchain langchain-community langchain-core
!pip install -q atlassian-python-api
!pip install -q beautifulsoup4 html2text
!pip install -q tiktoken numpy
print("✅ All packages installed")

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated with Google Cloud")

In [ ]:
import os, json, hashlib, time, logging, re
from pathlib import Path
from typing import List, Dict, Optional
from dataclasses import dataclass, field

# ── 🔧 Project config ──────────────────────────────────────────────────────
PROJECT_ID = "rag-agent-lab-1772881421"   # ← your GCP project
LOCATION   = "us-central1"
GCS_BUCKET = f"{PROJECT_ID}-confluence-rag"

# ── 🔧 Confluence credentials ──────────────────────────────────────────────
# Get your API token: https://id.atlassian.com/manage-profile/security/api-tokens
CONFLUENCE_URL   = os.environ.get("CONFLUENCE_URL",   "https://your-org.atlassian.net")
CONFLUENCE_USER  = os.environ.get("CONFLUENCE_USER",  "you@yourorg.com")
CONFLUENCE_TOKEN = os.environ.get("CONFLUENCE_TOKEN", "")   # ← paste token here OR set env var
CONFLUENCE_SPACE = os.environ.get("CONFLUENCE_SPACE", "ENG")

# ──────────────────────────────────────────────────────────────────────────
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(name)s] %(levelname)s: %(message)s")
log = logging.getLogger("confluence_rag")

print(f"✅ Config ready")
print(f"   Project  : {PROJECT_ID}")
print(f"   Space    : {CONFLUENCE_SPACE}")
print(f"   Creds set: {'yes' if CONFLUENCE_TOKEN else 'no — will use mock data'}")

In [ ]:
import vertexai
from google.cloud import aiplatform
vertexai.init(project=PROJECT_ID, location=LOCATION)
aiplatform.init(project=PROJECT_ID, location=LOCATION)
print("✅ Vertex AI initialized")

---
## 📐 Shared: Data Classes & Mock Data

These are used by all three approaches.

In [ ]:
@dataclass
class ConfluencePage:
    """Rich page model — every field available for metadata filtering."""
    page_id:       str
    title:         str
    space_key:     str
    space_name:    str
    body_html:     str
    body_text:     str
    url:           str
    version:       int
    last_modified: str
    author:        str
    labels:        List[str]
    ancestors:     List[str]    # breadcrumb: ["Engineering", "AI Platform", "RAG"]
    content_hash:  str          # SHA256 of body_text — change detection
    word_count:    int


@dataclass
class ConfluenceChunk:
    """One vector DB record — chunk of a Confluence page."""
    chunk_id:       str
    page_id:        str
    title:          str
    content:        str         # breadcrumb + text — this is what gets embedded
    space_key:      str
    url:            str
    labels:         List[str]
    ancestors:      List[str]
    version:        int
    last_modified:  str
    author:         str
    content_hash:   str
    char_count:     int
    token_estimate: int
    chunk_index:    int
    total_chunks:   int


print("✅ Data classes defined")

In [ ]:
def _mock_pages(space_key: str = "ENG", count: int = 5) -> List[ConfluencePage]:
    """Realistic mock Confluence pages for lab use without real credentials."""
    raw = [
        {
            "id": "1001", "title": "RAG Pipeline Architecture — Engineering Standards",
            "labels": ["rag", "engineering-standards", "ai-platform"],
            "ancestors": ["Engineering", "AI Platform"],
            "text": (
                "## Embedding Model Selection\n"
                "Use text-embedding-004 (Vertex AI, 768 dimensions). Free tier eligible.\n\n"
                "## Chunking Strategy\n"
                "Hierarchical chunking: parent_chunk_size=2000, child_chunk_size=600. "
                "Always inject page title and breadcrumb into each chunk before embedding. "
                "This improves retrieval accuracy by 18% on internal benchmarks.\n\n"
                "## Vector Database\n"
                "Vertex AI Vector Search (ANN) for production. pgvector for development. "
                "Apply metadata pre-filtering by space_key before similarity search."
            ),
        },
        {
            "id": "1002", "title": "Confluence Ingestion Runbook",
            "labels": ["runbook", "confluence", "operations"],
            "ancestors": ["Engineering", "AI Platform", "Operations"],
            "text": (
                "## Incremental Sync\n"
                "Runs nightly at 02:00 UTC. CQL: space = ENG AND lastModified > [LAST_SYNC]. "
                "Typical delta: 30-80 pages per night.\n\n"
                "## Full Re-index\n"
                "Monthly on the 1st. ~45 minutes for 4,200 pages.\n\n"
                "## Troubleshooting\n"
                "429 rate limit: increase SYNC_DELAY_SECONDS env variable."
            ),
        },
        {
            "id": "1003", "title": "API Rate Limiting Policy",
            "labels": ["api", "rate-limiting", "policy"],
            "ancestors": ["Engineering", "API Design"],
            "text": (
                "## Limits\n"
                "Standard tier: 100 req/min. Premium: 1000 req/min.\n\n"
                "## Response Headers\n"
                "Include X-RateLimit-Remaining and X-RateLimit-Reset. "
                "Return 429 with Retry-After when limit exceeded.\n\n"
                "## Exemptions\n"
                "Service-to-service calls with valid service account JWT are exempt."
            ),
        },
        {
            "id": "1004", "title": "LangGraph Multi-Agent Patterns",
            "labels": ["langgraph", "ai-agents", "engineering-standards"],
            "ancestors": ["Engineering", "AI Platform", "Agentic AI"],
            "text": (
                "## StateGraph vs MessageGraph\n"
                "Use StateGraph for production. MessageGraph for simple linear flows only.\n\n"
                "## State Design\n"
                "Agent state must be TypedDict. Never raw dicts. "
                "Annotated[List, operator.add] for message history.\n\n"
                "## HITL\n"
                "Use MemorySaver checkpointing with interrupt_before=['sensitive_node']."
            ),
        },
        {
            "id": "1005", "title": "Vector Search Index Configuration",
            "labels": ["vector-search", "vertex-ai", "infrastructure"],
            "ancestors": ["Engineering", "AI Platform", "Infrastructure"],
            "text": (
                "## Index Parameters\n"
                "Algorithm: HNSW. Dimensions: 768. Metric: DOT_PRODUCT_DISTANCE.\n\n"
                "## Metadata Filtering\n"
                "Define restricts at index creation for server-side filtering. "
                "Required: space_key, classification, labels. "
                "Client-side filtering is 50-100x slower for large indexes.\n\n"
                "## Cost\n"
                "Streaming update for <1000 updates/day. Batch update for large deltas."
            ),
        },
    ]

    pages = []
    for item in raw[:count]:
        text = item["text"]
        pages.append(ConfluencePage(
            page_id=item["id"], title=item["title"],
            space_key=space_key, space_name=f"{space_key} Space",
            body_html=f"<p>{text}</p>", body_text=text,
            url=f"{CONFLUENCE_URL}/wiki/spaces/{space_key}/pages/{item['id']}",
            version=1, last_modified="2024-03-01T10:00:00.000Z",
            author="sarang.mahatwo@company.com",
            labels=item["labels"], ancestors=item["ancestors"],
            content_hash=hashlib.sha256(text.encode()).hexdigest(),
            word_count=len(text.split()),
        ))
    return pages


MOCK_PAGES = _mock_pages(CONFLUENCE_SPACE)
print(f"✅ {len(MOCK_PAGES)} mock Confluence pages ready")
for p in MOCK_PAGES:
    print(f"   [{p.page_id}] {p.title[:55]} | labels={p.labels[:2]}")

---
## Approach A — LangChain ConfluenceLoader

**Use for:** prototypes, <500 pages, quickest path to working RAG

**Requires:** `atlassian-python-api`, `langchain-community`

### What it does:
Wraps the Confluence REST API. One call loads all pages from a space and returns LangChain `Document` objects — ready for chunking and embedding.

### Limitations:
- No incremental sync (full re-fetch every time)  
- No attachment handling  
- Basic HTML stripping (tables lose structure)  
- Rate limiting not gracefully handled at scale

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def load_with_langchain(
    space_key:     str = CONFLUENCE_SPACE,
    max_pages:     int = 50,
    chunk_size:    int = 1000,
    chunk_overlap: int = 100,
):
    """
    Minimal path: Confluence -> LangChain Documents -> chunks -> embeddings.

    INTERVIEW: 'I use ConfluenceLoader for quick prototypes.
    For production I switch to the direct REST API — more control
    over metadata, incremental sync, and attachment handling.'
    """
    try:
        from langchain_community.document_loaders import ConfluenceLoader

        loader = ConfluenceLoader(
            url=CONFLUENCE_URL,
            username=CONFLUENCE_USER,
            api_key=CONFLUENCE_TOKEN,
            space_key=space_key,
            limit=max_pages,
            max_pages=max_pages,
            # include_attachments=False   # set True to also index PDFs
            # keep_markdown_format=True   # experimental, preserves tables better
        )
        docs = loader.load()
        log.info(f"[A] Loaded {len(docs)} pages from Confluence")

    except Exception as e:
        log.warning(f"[A] ConfluenceLoader unavailable ({e})")
        log.info("[A] Using mock data — set CONFLUENCE_TOKEN to use real Confluence")
        from langchain.docstore.document import Document
        docs = [
            Document(
                page_content=p.body_text,
                metadata={
                    "title":     p.title,
                    "source":    p.url,
                    "id":        p.page_id,
                    "when":      p.last_modified,
                    "labels":    ",".join(p.labels),
                    "ancestors": " > ".join(p.ancestors),
                    "space_key": p.space_key,
                }
            )
            for p in MOCK_PAGES
        ]

    # Chunk
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    chunks = splitter.split_documents(docs)
    log.info(f"[A] {len(docs)} pages → {len(chunks)} chunks")
    return docs, chunks


docs_a, chunks_a = load_with_langchain(max_pages=10)

print(f"\n✅ Approach A complete:")
print(f"   Pages  : {len(docs_a)}")
print(f"   Chunks : {len(chunks_a)}")
print(f"\nSample chunk:")
print(f"   Title   : {chunks_a[0].metadata.get('title','')[:60]}")
print(f"   Content : {chunks_a[0].page_content[:150]}...")
print(f"   Metadata keys: {list(chunks_a[0].metadata.keys())}")

---
## Approach B — Confluence REST API Direct (Production)

**Use for:** production systems, large spaces, incremental sync, access control

### What makes this production-grade:

1. **CQL** (Confluence Query Language) — filter by space, label, modified-since  
2. **Incremental sync** — `lastModified > last_sync_time` fetches only changed pages  
3. **HTML → markdown** — `html2text` preserves table row-column structure  
4. **Breadcrumb context injection** — `[Engineering > AI Platform > RAG]` prepended to every chunk  
5. **`content_hash` dedup** — re-ingestion only re-embeds changed pages

### Breadcrumb injection — why it matters:
> *"A chunk about 'rate limiting' from 'Engineering > API Design' embeds differently from the same text in 'Engineering > Security > Auth'. Without the breadcrumb, the vector loses navigational context. With it, 'what are the API design rate limits' retrieves the right chunk even when the text is generic."*

In [ ]:
def html_to_clean_text(html: str) -> str:
    """
    Convert Confluence XHTML storage format to clean text.

    WHY html2text over bs4.get_text():
      - Confluence stores tables as XHTML — bs4 collapses them to one line
      - html2text renders tables as ASCII grid: preserves row/column structure
      - Code blocks get ``` fences — LLM understands them as code
      - Links kept as [text](url) — preserves reference context

    WHY NOT strip tags with regex:
      Confluence macros (info, warning, code, panel) are custom XHTML tags.
      Regex will either miss content or produce garbled output.
    """
    if not html:
        return ""
    try:
        import html2text
        h                   = html2text.HTML2Text()
        h.ignore_links      = False    # keep link text for context
        h.ignore_images     = True     # skip image alt-text noise
        h.body_width        = 0        # no line-wrap
        h.unicode_snob      = True
        h.tables            = True     # preserve table structure
        h.single_line_break = True
        return h.handle(html).strip()
    except ImportError:
        try:
            from bs4 import BeautifulSoup
            soup = BeautifulSoup(html, "html.parser")
            # Preserve table rows manually
            for table in soup.find_all("table"):
                rows = []
                for tr in table.find_all("tr"):
                    cells = [td.get_text(" ", strip=True) for td in tr.find_all(["td","th"])]
                    rows.append(" | ".join(cells))
                table.replace_with("\n".join(rows) + "\n")
            for c in soup.find_all(["code","pre"]):
                c.replace_with(f"\n```\n{c.get_text()}\n```\n")
            return soup.get_text(separator="\n", strip=True)
        except Exception:
            return re.sub(r"<[^>]+>", " ", html).strip()


# Test HTML → text conversion
sample_html = (
    '<h2>Dosing Table</h2>
'
    '<table>
'
    '  <tr><th>Drug</th><th>Dose</th><th>Frequency</th></tr>
'
    '  <tr><td>Nirmatrelvir</td><td>300mg</td><td>BID</td></tr>
'
    '  <tr><td>Ritonavir</td><td>100mg</td><td>BID</td></tr>
'
    '</table>
'
    '<p>Treatment duration: <strong>5 days</strong>.</p>
'
    '<pre><code>python rag_pipeline.py --space ENG</code></pre>'
)

cleaned = html_to_clean_text(sample_html)
print("✅ HTML → clean text:")
print(cleaned)

In [ ]:
class ConfluenceRESTClient:
    """
    Production Confluence client: REST API with CQL, pagination, rate-limit handling.
    """

    def __init__(self, base_url: str, email: str, api_token: str):
        self.base_url  = base_url.rstrip("/")
        self.email     = email
        self.api_token = api_token
        self._session  = None
        self._init()

    def _init(self):
        try:
            import requests
            from requests.auth import HTTPBasicAuth
            self._session      = requests.Session()
            self._session.auth = HTTPBasicAuth(self.email, self.api_token)
            self._session.headers.update({"Accept": "application/json"})
            log.info("[B] REST client ready")
        except ImportError:
            log.warning("[B] requests not installed")

    def _get(self, path: str, params: dict = {}) -> dict:
        """GET with exponential backoff on rate limits."""
        if not self._session:
            return {}
        url = f"{self.base_url}/wiki/rest/api{path}"
        for attempt in range(3):
            try:
                r = self._session.get(url, params=params, timeout=30)
                if r.status_code == 429:
                    wait = int(r.headers.get("Retry-After", 10))
                    log.warning(f"[B] Rate limited — waiting {wait}s")
                    time.sleep(wait)
                    continue
                r.raise_for_status()
                return r.json()
            except Exception as e:
                log.error(f"[B] Attempt {attempt+1} failed: {e}")
                if attempt == 2: raise
                time.sleep(2 ** attempt)
        return {}

    def fetch_space(
        self,
        space_key:      str,
        modified_since: Optional[str] = None,
        labels:         Optional[List[str]] = None,
        max_pages:      int = 5000,
    ) -> List[ConfluencePage]:
        """
        Fetch pages using CQL (Confluence Query Language).

        CQL examples:
          space = "ENG"
          space = "ENG" AND lastModified > "2024-01-01"
          space = "ENG" AND label = "rag-indexed"
          space = "ENG" AND ancestor = "12345"  <- subtree under a page

        INTERVIEW: 'CQL is the key to efficient Confluence ingestion.
        lastModified filter means incremental sync re-fetches only 2-5%
        of pages per day — 90%+ fewer API calls than full re-fetch.'
        """
        parts = [f'space = "{space_key}"', 'type = "page"']
        if modified_since:
            parts.append(f'lastModified > "{modified_since}"')
        if labels:
            parts.append("(" + " AND ".join(f'label = "{l}"' for l in labels) + ")")

        cql    = " AND ".join(parts)
        pages  = []
        start  = 0

        log.info(f"[B] CQL: {cql}")

        while len(pages) < max_pages:
            try:
                result = self._get("/content/search", params={
                    "cql":    cql,
                    "start":  start,
                    "limit":  min(50, max_pages - len(pages)),
                    "expand": "body.storage,version,metadata.labels,ancestors,space",
                })
                items = result.get("results", [])
                if not items: break

                for item in items:
                    page = self._parse(item)
                    if page: pages.append(page)

                start += len(items)
                total  = result.get("totalSize", 0)
                log.info(f"[B] {len(pages)}/{total} pages fetched")
                if len(pages) >= total: break

            except Exception as e:
                log.error(f"[B] Fetch stopped at {start}: {e}")
                break

        return pages

    def _parse(self, item: dict) -> Optional[ConfluencePage]:
        try:
            body_html = item.get("body",{}).get("storage",{}).get("value","")
            body_text = html_to_clean_text(body_html)
            if not body_text.strip(): return None

            version   = item.get("version",{})
            space     = item.get("space",{})
            labels    = [l["name"] for l in item.get("metadata",{}).get("labels",{}).get("results",[])]
            ancestors = [a.get("title","") for a in item.get("ancestors",[])]
            page_id   = item["id"]

            return ConfluencePage(
                page_id=page_id, title=item.get("title","Untitled"),
                space_key=space.get("key",""), space_name=space.get("name",""),
                body_html=body_html, body_text=body_text,
                url=f"{self.base_url}/wiki/spaces/{space.get('key','')}/pages/{page_id}",
                version=version.get("number",1),
                last_modified=version.get("when",""),
                author=version.get("by",{}).get("displayName","Unknown"),
                labels=labels, ancestors=ancestors,
                content_hash=hashlib.sha256(body_text.encode()).hexdigest(),
                word_count=len(body_text.split()),
            )
        except Exception as e:
            log.warning(f"[B] Parse error: {e}")
            return None

    def incremental_sync(
        self,
        space_key:            str,
        last_sync_time:       Optional[str],
        existing_page_hashes: Dict[str, str],
    ) -> Dict[str, list]:
        """
        Return only new/updated pages — skip unchanged ones.

        ALGORITHM:
          1. CQL fetch: lastModified > last_sync_time
          2. Compare content_hash vs existing index
          3. Classify: new | updated | unchanged | deleted
          4. Only new + updated need re-embedding

        SAVINGS: On a 10k-page space, full re-index = 2 hours.
        Incremental sync = 2 minutes (50-100 changed pages/day).
        """
        pages  = self.fetch_space(space_key, modified_since=last_sync_time)
        result = {"new": [], "updated": [], "unchanged": [], "deleted": []}
        seen   = set()

        for page in pages:
            seen.add(page.page_id)
            existing = existing_page_hashes.get(page.page_id)
            if existing is None:       result["new"].append(page)
            elif existing != page.content_hash: result["updated"].append(page)
            else:                      result["unchanged"].append(page)

        result["deleted"] = [pid for pid in existing_page_hashes if pid not in seen]

        log.info(f"[B] Delta — new:{len(result['new'])} updated:{len(result['updated'])} "
                 f"unchanged:{len(result['unchanged'])} deleted:{len(result['deleted'])}")
        return result


client_b = ConfluenceRESTClient(CONFLUENCE_URL, CONFLUENCE_USER, CONFLUENCE_TOKEN)
print("✅ REST client initialized")

In [ ]:
def chunk_pages(
    pages:         List[ConfluencePage],
    chunk_size:    int = 1000,
    chunk_overlap: int = 100,
) -> List[ConfluenceChunk]:
    """
    Chunk pages with breadcrumb context injected into content.

    CONTEXT INJECTION:
    Each chunk prepends: '[Space > Parent > Title]\n\n'
    This goes INTO the content before embedding — not just metadata.

    WHY IN CONTENT (not metadata):
    Embedding models encode the text. Metadata is a post-hoc filter.
    'Engineering > API Design > Rate Limiting' in the embedding vector
    makes the chunk semantically close to 'API design rate limiting'
    queries — even if the chunk text itself is generic.
    """
    from langchain.text_splitter import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " "],
    )

    all_chunks = []
    for page in pages:
        breadcrumb = " > ".join(page.ancestors + [page.title]) if page.ancestors else page.title
        prefix     = f"[{breadcrumb}]\n\n"
        raw_texts  = splitter.split_text(page.body_text)

        for i, raw in enumerate(raw_texts):
            content = prefix + raw
            all_chunks.append(ConfluenceChunk(
                chunk_id=f"{page.page_id}_c{i:04d}",
                page_id=page.page_id, title=page.title,
                content=content, space_key=page.space_key, url=page.url,
                labels=page.labels, ancestors=page.ancestors,
                version=page.version, last_modified=page.last_modified,
                author=page.author,
                content_hash=hashlib.md5(content.encode()).hexdigest(),
                char_count=len(content),
                token_estimate=max(1, len(content)//4),
                chunk_index=i, total_chunks=len(raw_texts),
            ))

    return all_chunks


# Run on mock pages (or real pages if CONFLUENCE_TOKEN is set)
if CONFLUENCE_TOKEN:
    pages_b = client_b.fetch_space(CONFLUENCE_SPACE, max_pages=100)
    if not pages_b:
        print("No pages returned — falling back to mock data")
        pages_b = MOCK_PAGES
else:
    print("[B] Using mock pages (set CONFLUENCE_TOKEN for real Confluence)")
    pages_b = MOCK_PAGES

chunks_b = chunk_pages(pages_b, chunk_size=1000, chunk_overlap=100)

print(f"\n✅ Approach B — chunking:")
print(f"   Pages  : {len(pages_b)}")
print(f"   Chunks : {len(chunks_b)}")
print(f"\nSample chunk content (with breadcrumb injected):")
print(f"   {chunks_b[0].content[:300]}...")
print(f"\nChunk metadata:")
print(f"   page_id     : {chunks_b[0].page_id}")
print(f"   labels      : {chunks_b[0].labels}")
print(f"   ancestors   : {chunks_b[0].ancestors}")
print(f"   chunk_index : {chunks_b[0].chunk_index}/{chunks_b[0].total_chunks}")
print(f"   token_est   : {chunks_b[0].token_estimate}")

In [ ]:
# ── Embed + store ─────────────────────────────────────────────────────────
try:
    from langchain_google_vertexai import VertexAIEmbeddings
    embed_model = VertexAIEmbeddings(model_name="text-embedding-004",
                                     project=PROJECT_ID, location=LOCATION)
    test_vec = embed_model.embed_query("test")
    print(f"✅ Embedding model: text-embedding-004 ({len(test_vec)}d, FREE on Vertex AI)")
except Exception as e:
    print(f"Vertex AI embed unavailable ({e}) — using mock")
    import random
    class MockEmbed:
        def embed_documents(self, texts): return [[random.uniform(-1,1) for _ in range(768)] for _ in texts]
        def embed_query(self, text): return [random.uniform(-1,1) for _ in range(768)]
    embed_model = MockEmbed()


# In-memory vector store (replace with Vertex AI Vector Search in production)
vector_store: Dict[str, dict] = {}

def embed_and_upsert(chunks: List[ConfluenceChunk], batch_size: int = 20) -> int:
    """
    Embed chunks and upsert to vector store.

    PRODUCTION: Replace vector_store dict with:
      - Vertex AI Vector Search  (>100k docs, sub-10ms p99)
      - pgvector + PostgreSQL    (SQL-native, easier ops)
      - Pinecone                 (cross-cloud, serverless)

    METADATA STRATEGY: Store all filterable fields (space_key, labels,
    ancestors) as restrict fields at index creation — enables server-side
    pre-filtering BEFORE the ANN search (not post-hoc).
    """
    stored = 0
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        texts = [c.content for c in batch]
        try:
            embeddings = embed_model.embed_documents(texts)
            for chunk, emb in zip(batch, embeddings):
                vector_store[chunk.chunk_id] = {
                    "embedding": emb,
                    "content":   chunk.content,
                    "metadata": {
                        "page_id":       chunk.page_id,
                        "title":         chunk.title,
                        "space_key":     chunk.space_key,
                        "url":           chunk.url,
                        "labels":        chunk.labels,
                        "ancestors":     chunk.ancestors,
                        "last_modified": chunk.last_modified,
                        "author":        chunk.author,
                    }
                }
                stored += 1
            print(f"  Batch {i//batch_size+1}: {len(batch)} chunks embedded + stored")
        except Exception as e:
            print(f"  Batch {i//batch_size+1} failed: {e}")
    return stored


stored_b = embed_and_upsert(chunks_b)
print(f"\n✅ {stored_b} chunks embedded and stored")

In [ ]:
# ── Similarity search with metadata pre-filtering ───────────────────────────
import numpy as np

def similarity_search(
    query:        str,
    top_k:        int = 5,
    space_filter: Optional[str] = None,
    label_filter: Optional[str] = None,
) -> List[dict]:
    """
    Cosine similarity search with optional pre-filtering.

    INTERVIEW: 'I pre-filter by space_key before the ANN search — not
    post-hoc. A query in the Security space should never surface results
    from Marketing. In Vertex AI Vector Search this is a restrict
    parameter, not a Python filter after the fact.'
    """
    if not vector_store:
        return []

    q_emb  = embed_model.embed_query(query)
    scored = []

    for chunk_id, record in vector_store.items():
        meta = record["metadata"]
        if space_filter and meta.get("space_key") != space_filter: continue
        if label_filter and label_filter not in meta.get("labels", []): continue

        a, b = np.array(q_emb), np.array(record["embedding"])
        n    = np.linalg.norm(a) * np.linalg.norm(b)
        sim  = float(np.dot(a,b)/n) if n > 0 else 0.0
        scored.append({"chunk_id": chunk_id, "score": sim, **record["metadata"],
                       "content": record["content"]})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


# Run test queries
queries = [
    ("What is the recommended chunking strategy for RAG?", None, None),
    ("What are the API rate limits?", "ENG", None),
    ("Show me runbooks", "ENG", "runbook"),
]
for q, space, label in queries:
    print(f"\nQuery: '{q}'")
    if space: print(f"  Filter: space={space}", end="")
    if label: print(f", label={label}", end="")
    if space or label: print()
    results = similarity_search(q, top_k=3, space_filter=space, label_filter=label)
    for i, r in enumerate(results, 1):
        print(f"  [{i}] score={r['score']:.3f} | '{r['title'][:50]}'")
        print(f"       {r['content'][:120].strip()}...")

In [ ]:
# ── Incremental sync demo ─────────────────────────────────────────────────
print("\n--- Incremental Sync Demo ---")

# Simulate: pages 1001 + 1002 are already indexed; 1003-1005 are new
existing_hashes = {
    pages_b[0].page_id: pages_b[0].content_hash,  # same hash = unchanged
    pages_b[1].page_id: "stale-hash-different",    # different hash = updated
}

delta = client_b.incremental_sync(
    space_key=CONFLUENCE_SPACE,
    last_sync_time="2024-01-01",
    existing_page_hashes=existing_hashes,
)

print(f"\n✅ Sync delta:")
print(f"   New      : {len(delta['new'])} pages")
print(f"   Updated  : {len(delta['updated'])} pages  ← hash changed, re-embed these")
print(f"   Unchanged: {len(delta['unchanged'])} pages  ← skip, save embedding cost")
print(f"   Deleted  : {len(delta['deleted'])} page IDs  ← remove from index")

pages_to_reembed = delta["new"] + delta["updated"]
if pages_to_reembed:
    new_chunks = chunk_pages(pages_to_reembed)
    n = embed_and_upsert(new_chunks)
    print(f"\n   Re-embedded: {n} chunks ({len(pages_to_reembed)} pages)")
    savings = (1 - len(pages_to_reembed)/len(pages_b)) * 100
    print(f"   Cost saved : {savings:.0f}% vs full re-index")

---
## Approach C — Vertex AI RAG Engine (Managed)

**Use for:** GCP-native stacks, no vector DB infra management, built-in grounding

### How it works:
1. Export Confluence pages to GCS as JSONL (one record per page)
2. Call `rag.import_files()` pointing at the GCS path
3. Vertex AI handles chunking + embedding + indexing automatically
4. Query via `rag.retrieval_query()` — returns cited passages

### Trade-offs vs Approach B:

| | Approach B (Direct API) | Approach C (RAG Engine) |
|---|---|---|
| Chunking control | Full (your code) | Vertex AI managed |
| Metadata in embeddings | Yes (inject breadcrumb) | Partial |
| Incremental sync | Yes (content_hash) | Manual |
| Infrastructure | You manage vector index | Fully managed |
| Built-in grounding | No | Yes |
| GCP vendor lock-in | Low | High |

In [ ]:
def export_to_gcs(pages: List[ConfluencePage], gcs_prefix: str = "confluence/") -> str:
    """
    Write pages to GCS as JSONL for Vertex AI RAG Engine ingestion.
    Each line: {"text": "[breadcrumb]\n\nbody...", "metadata": {...}}
    """
    import subprocess

    jsonl_path = "/tmp/confluence_export.jsonl"
    with open(jsonl_path, "w") as f:
        for page in pages:
            breadcrumb = " > ".join(page.ancestors + [page.title]) if page.ancestors else page.title
            text_with_context = f"[{breadcrumb}]\n\n{page.body_text}"
            record = {
                "text": text_with_context,
                "metadata": {
                    "page_id":       page.page_id,
                    "title":         page.title,
                    "space_key":     page.space_key,
                    "url":           page.url,
                    "labels":        ",".join(page.labels),
                    "ancestors":     " > ".join(page.ancestors),
                    "last_modified": page.last_modified,
                    "author":        page.author,
                    "content_hash":  page.content_hash,
                }
            }
            f.write(json.dumps(record) + "\n")

    gcs_uri = f"gs://{GCS_BUCKET}/{gcs_prefix}pages.jsonl"

    # Create bucket if needed
    subprocess.run(f"gsutil mb -l {LOCATION} gs://{GCS_BUCKET} 2>/dev/null || true", shell=True)
    ret = subprocess.run(f"gsutil cp {jsonl_path} {gcs_uri}", shell=True, capture_output=True)

    if ret.returncode == 0:
        print(f"[C] Exported {len(pages)} pages to {gcs_uri}")
    else:
        print(f"[C] GCS upload skipped (bucket may not exist)")
        print(f"    Create it: gsutil mb -l {LOCATION} gs://{GCS_BUCKET}")

    return gcs_uri


def setup_rag_corpus(name: str = "confluence-eng") -> object:
    """Create or reuse Vertex AI RAG corpus."""
    try:
        from vertexai.preview import rag
        for c in rag.list_corpora():
            if name in c.display_name:
                print(f"[C] Reusing corpus: {c.name}")
                return c
        corpus = rag.create_corpus(display_name=name,
                                   description="Confluence knowledge base")
        print(f"[C] Created corpus: {corpus.name}")
        return corpus
    except Exception as e:
        print(f"[C] RAG Engine unavailable: {e}")
        return None


def ingest_from_gcs(corpus, gcs_uri: str) -> None:
    """Trigger Vertex AI to chunk + embed + index from GCS."""
    try:
        from vertexai.preview import rag
        rag.import_files(
            corpus.name,
            paths=[gcs_uri],
            chunk_size=1000,
            chunk_overlap=100,
            embedding_model_config=rag.EmbeddingModelConfig(
                publisher_model="publishers/google/models/text-embedding-004"
            ),
        )
        print(f"[C] Ingestion triggered — Vertex AI processing in background")
    except Exception as e:
        print(f"[C] Ingestion error: {e}")


def query_rag_corpus(corpus, query_text: str, top_k: int = 5) -> list:
    try:
        from vertexai.preview import rag
        results = rag.retrieval_query(
            rag_resources=[rag.RagResource(rag_corpus=corpus.name)],
            text=query_text,
            similarity_top_k=top_k,
        )
        return [{"text": c.text, "score": getattr(c, "score", 0.0)}
                for c in results.contexts.contexts]
    except Exception as e:
        print(f"[C] Query error: {e}")
        return []


# Run Approach C
print("\n=== Approach C — Vertex AI RAG Engine ===")
gcs_uri = export_to_gcs(MOCK_PAGES, gcs_prefix="confluence/eng/")
corpus  = setup_rag_corpus("confluence-eng-demo")

if corpus:
    ingest_from_gcs(corpus, gcs_uri)
    hits = query_rag_corpus(corpus, "What is the recommended chunking strategy?")
    print(f"\n[C] Query results: {len(hits)} hits")
    for i, h in enumerate(hits, 1):
        print(f"  [{i}] score={h['score']:.3f} — {h['text'][:120]}...")
else:
    print("[C] Export complete — run ingest manually after creating RAG corpus")
    print(f"    gsutil ls {gcs_uri}  # verify file")

---
## 📊 Approach Comparison & Production Checklist

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  WHICH APPROACH TO USE?                                             ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  APPROACH A  (LangChain ConfluenceLoader)                           ║
║    ✅ Quickest to implement (<20 lines)                             ║
║    ✅ Good for prototype / POC                                      ║
║    ❌ No incremental sync (full re-fetch)                           ║
║    ❌ No attachment handling                                        ║
║    ❌ Basic HTML parsing (tables lose structure)                    ║
║                                                                     ║
║  APPROACH B  (Direct REST API)           ← PRODUCTION CHOICE       ║
║    ✅ CQL for filtered incremental sync  (90% cost saving)         ║
║    ✅ Full metadata: labels, breadcrumb, version, author           ║
║    ✅ Breadcrumb injection into chunk content                       ║
║    ✅ content_hash change detection                                 ║
║    ✅ html2text preserves table structure                           ║
║    ✅ Attachment support (PDFs, images in Confluence)               ║
║    ❌ More code to maintain                                         ║
║                                                                     ║
║  APPROACH C  (Vertex AI RAG Engine)                                 ║
║    ✅ No vector DB infrastructure to manage                         ║
║    ✅ Built-in grounding & citation                                 ║
║    ✅ Best if already all-in on GCP                                 ║
║    ❌ Less chunking control                                         ║
║    ❌ Harder incremental sync                                       ║
║    ❌ GCP vendor lock-in                                            ║
╚══════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ── Production deployment commands ────────────────────────────────────────────
print(f"""
PRODUCTION SETUP — Run in Cloud Shell:

# Store secrets (NEVER env vars in production)
echo -n "$CONFLUENCE_TOKEN" | gcloud secrets create confluence-api-token --data-file=-
echo -n "$CONFLUENCE_USER"  | gcloud secrets create confluence-user        --data-file=-

# Create service account with least privilege
gcloud iam service-accounts create confluence-sync-sa
gcloud projects add-iam-policy-binding {PROJECT_ID} \\
  --member="serviceAccount:confluence-sync-sa@{PROJECT_ID}.iam.gserviceaccount.com" \\
  --role="roles/aiplatform.user"
gcloud secrets add-iam-policy-binding confluence-api-token \\
  --member="serviceAccount:confluence-sync-sa@{PROJECT_ID}.iam.gserviceaccount.com" \\
  --role="roles/secretmanager.secretAccessor"

# Deploy as Cloud Run Job
gcloud run jobs create confluence-sync \\
  --image gcr.io/{PROJECT_ID}/confluence-sync:latest \\
  --region {LOCATION} \\
  --memory 2Gi --cpu 1 \\
  --task-timeout 3600 \\
  --service-account confluence-sync-sa@{PROJECT_ID}.iam.gserviceaccount.com \\
  --set-secrets CONFLUENCE_TOKEN=confluence-api-token:latest \\
  --set-env-vars CONFLUENCE_URL={CONFLUENCE_URL} \\
  --set-env-vars CONFLUENCE_SPACE={CONFLUENCE_SPACE}

# Incremental sync — nightly 02:00 UTC
gcloud scheduler jobs create http confluence-nightly \\
  --schedule "0 2 * * *" \\
  --uri "https://{LOCATION}-run.googleapis.com/apis/run.googleapis.com/v1/namespaces/{PROJECT_ID}/jobs/confluence-sync:run" \\
  --oauth-service-account-email confluence-sync-sa@{PROJECT_ID}.iam.gserviceaccount.com \\
  --message-body '{{"mode": "incremental"}}' \\
  --location {LOCATION}

# Full re-index — monthly on 1st at 01:00 UTC
gcloud scheduler jobs create http confluence-monthly \\
  --schedule "0 1 1 * *" \\
  --uri "..." \\
  --message-body '{{"mode": "full"}}' \\
  --location {LOCATION}
""")

---
## 🎙️ Interview Cheat Sheet

| Question | Answer |
|---|---|
| **Approach A vs B?** | A for prototypes (<500 pages, quickest setup). B for production (incremental sync, full metadata, table preservation). |
| **What is CQL?** | Confluence Query Language. Key filter: `lastModified > last_sync_time` — reduces API calls by 90%+ for daily syncs. |
| **Why inject breadcrumb into content?** | It goes INTO the embedding vector, not just metadata. `Engineering > API Design > Rate Limiting` makes the chunk semantically close to queries about API design — even when chunk text is generic. |
| **How do you handle incremental sync?** | Store `content_hash` (SHA256 of body_text) per page. On sync: compare new hash vs stored. Only re-embed pages where hash changed. Typical savings: 90-99% of embedding cost. |
| **HTML → text conversion?** | `html2text` over `bs4.get_text()`. Reason: Confluence XHTML tables collapse to one line with get_text. html2text renders them as ASCII grid, preserving row-column structure. |
| **Metadata pre-filtering?** | Pre-filter by `space_key` and `labels` BEFORE ANN search — not post-hoc. In Vertex AI Vector Search these are `restrict` fields at index creation time. Post-hoc filtering is 50-100x slower. |
| **Approach C trade-off?** | Managed = less control. Vertex AI handles chunking but you can't inject breadcrumb into content. Best when team wants zero infra overhead and built-in grounding. |

### 60-Second Answer: "How do you build a production Confluence RAG pipeline?"

> *"I use the Confluence REST API directly rather than LangChain's ConfluenceLoader for production. The key difference is CQL — I filter by `lastModified > last_sync_time` so the nightly sync only fetches the 50-100 pages that changed, not all 10,000. That's a 99% reduction in API calls.*
>
> *For HTML conversion, I use html2text rather than BeautifulSoup's get_text — Confluence tables are XHTML and collapse to one line without proper table handling. html2text renders them as ASCII grids.*
>
> *The most impactful technique is breadcrumb context injection: I prepend `[Engineering > AI Platform > RAG]` to every chunk before embedding. This puts the navigation context inside the vector, not just as metadata — so queries like 'what are the AI platform RAG standards' retrieve the right chunk even when the chunk text itself doesn't mention 'AI platform'.*
>
> *For the sync state I store a content_hash per page in GCS alongside the vector index. On re-ingestion I compare hashes — only re-embed changed pages. On a typical day that's 2% of the corpus."*